# 3주차 실습: AI 에이전트 위키 문서로 Reranking 전후 비교

문서: 한국어 위키백과 「AI 에이전트」 (본문 약 3만 자)

흐름
1. 위키 문서를 불러와서 잡음을 지우고 **txt로 저장해 고정**
2. 청킹 → 임베딩 → Chroma 벡터 DB
3. 기존 방식(벡터 검색 상위 4개)과 리랭킹 방식(후보 20개 → 리랭커 상위 4개)을 비교

> Colab 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`를 추천합니다.

In [1]:
!pip install -q -U langchain langchain-community langchain-openai langchain-text-splitters chromadb beautifulsoup4 sentence-transformers

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["USER_AGENT"] = "DRAG-study-bot/0.1"   # WebBaseLoader 경고 제거용

## 1. 위키 문서 불러오기 → 정리 → txt로 저장

위키는 수시로 편집되므로, 한 번 불러온 내용을 파일로 저장해 두고 이후에는 그 파일만 씁니다.
**`ai_agent_wiki.txt`가 이미 있으면 이 셀은 다시 실행하지 않아도 됩니다.**

In [3]:
import re
import bs4
from langchain_community.document_loaders import WebBaseLoader

SOURCE_URL = "https://ko.wikipedia.org/wiki/AI_에이전트"

loader = WebBaseLoader(
    web_paths=(SOURCE_URL,),
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(id="mw-content-text")),  # 위키백과 본문 영역만(mw-content-text)
)
raw_text = loader.load()[0].page_content

text = raw_text
text = text.replace("[편집]", "")          # 소제목 옆 [편집] 링크 제거
text = re.sub(r"\[\d+\]", "", text)       # 각주 번호 [1], [23] 제거
text = re.sub(r"\[출처 필요\]", "", text)
cut = text.find("\n각주")                   # 문서 끝의 각주/참고문헌 목록 잘라내기
if cut > 0:
    text = text[:cut]
text = re.sub(r"\n{3,}", "\n\n", text).strip()

print("정리 전:", len(raw_text), "자 / 정리 후:", len(text), "자")
print("----- 마지막 300자 (각주 목록이 잘 잘렸는지 확인) -----")
print(text[-300:])

with open("ai_agent_wiki.txt", "w", encoding="utf-8") as f:
    f.write(text)

/tmp/ipykernel_6387/2713002966.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


정리 전: 31079 자 / 정리 후: 10347 자
----- 마지막 300자 (각주 목록이 잘 잘렸는지 확인) -----
 등 6가지 범주에서 위협을 식별하는 마이크로소프트의 모델이다.
MITRE ATLAS: AI 시스템에 대한 공격 전술 및 기술에 관한 지식 베이스다.
OWASP 생성형 AI 보안 프로젝트: 특히 에이전트형 애플리케이션을 위한 가이드라인을 포함하여, 생성형 AI 및 대형 언어 모델 통합과 관련된 보안 취약점에 대한 지침을 제공한다.
MAESTRO: AI 시스템의 생애 주기 전반에 걸쳐 위험을 평가하기 위한 클라우드 시큐리티 얼라이언스의 프레임워크다.
같이 보기
지능형 에이전트
모델 컨텍스트 프로토콜
합리적 에이전트
소프트웨어 에이전트


## 2. 저장한 txt로 기본 RAG 구성

In [4]:
from langchain_community.document_loaders import TextLoader

documents = TextLoader("ai_agent_wiki.txt", encoding="utf-8").load()

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # 3만 자 문서 → 청크 70~80개
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = text_splitter.split_documents(documents)

# txt에는 페이지 정보가 없으니 청크 번호를 출처로 사용
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

print("청크 수:", len(chunks))
print("===============================================================================================================================")
print(chunks[5].page_content)

청크 수: 34
원샷 모델 쿼리: 행동 계획을 생성하기 위해 모델에 한 번 쿼리하는 방식이다.
참조 아키텍처
켄 황(Ken Huang)은 각 계층이 아래 계층의 기능을 기반으로 구축되는 7개의 상호 연결된 계층으로 구성된 AI 에이전트 참조 아키텍처를 제안했다.:
1계층: 파운데이션 모델 - 에이전트 기능을 구동하기 위한 핵심 AI 엔진을 제공한다.
2계층: 데이터 운영 - 벡터 데이터베이스, 데이터 로더, RAG를 포함하여 AI 에이전트 운영에 필요한 복잡한 데이터 인프라를 관리한다.
3계층: 에이전트 프레임워크 - AI 에이전트의 개발 및 관리를 단순화하는 정교한 소프트웨어 및 도구다.
4계층: 전개 및 인프라 - AI 에이전트를 실행하기 위한 강력한 기술적 토대를 제공한다.


In [6]:
import shutil
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

DB_DIR = "./chroma_ai_agent"
shutil.rmtree(DB_DIR, ignore_errors=True)   # 다시 실행해도 중복이 쌓이지 않도록 매번 새로 만듦

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_DIR
)
print("벡터 DB 저장 수:", vectorstore._collection.count())  # 청크 수와 같아야 정상

벡터 DB 저장 수: 34


In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

prompt = ChatPromptTemplate.from_template("""
당신은 제공된 문서를 기반으로 질문에 답변하는 어시스턴트입니다.
아래 제공된 컨텍스트만을 사용하여 질문에 답변하세요.
컨텍스트에 답변할 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.
답변에 사용한 문서의 출처(청크 번호)를 명시하세요.

컨텍스트:
{context}

질문: {question}

답변:
""")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs_with_source(docs):
    return "\n\n".join(f"[청크 {d.metadata.get('chunk_id')}]\n{d.page_content}" for d in docs)

## 3. 기존 방식: 벡터 검색 상위 4개

In [8]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

baseline_chain = (
    {"context": retriever | format_docs_with_source, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 4. 리랭킹 방식: 후보 20개 → Cross-encoder로 다시 채점 → 상위 4개

`BAAI/bge-reranker-v2-m3`는 한국어를 지원하는 다국어 cross-encoder입니다. 처음 실행 시 약 2GB를 내려받습니다.
`reranker.predict([(질문, 문서), ...])`가 발표에서 본 `[CLS] 질문 [SEP] 문서 → 점수` 구조입니다.

In [9]:
FIRST_STAGE_K = 20   # 1단계 후보 수
FINAL_TOP_N = 4       # 최종 문서 수 (기존 방식의 k=4와 맞춤)

base_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": FIRST_STAGE_K})

In [10]:
import torch
from sentence_transformers import CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=device)
print("device:", device)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

device: cuda


In [11]:
def rerank(query, docs, top_n=FINAL_TOP_N):
    pairs = [(query, d.page_content) for d in docs]      # 질문-문서 쌍
    scores = reranker.predict(pairs)                     # 쌍마다 관련성 점수
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    results = []
    for doc, score in ranked[:top_n]:
        doc.metadata["rerank_score"] = float(score)
        results.append(doc)
    return results

def retrieve_and_rerank(query):
    return rerank(query, base_retriever.invoke(query))

rerank_chain = (
    {"context": RunnableLambda(retrieve_and_rerank) | format_docs_with_source, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 5. 검색 결과 전후 비교

"원래 몇 위였던 청크가 올라왔는지"와 "리랭킹에 걸린 시간"을 같이 봅니다.

In [12]:
import time

def _preview(text, n=70):
    return text[:n].replace("\n", " ")

def compare(query, top_n=FINAL_TOP_N):
    t0 = time.perf_counter()
    candidates = base_retriever.invoke(query)
    t1 = time.perf_counter()
    reranked = rerank(query, candidates, top_n)
    t2 = time.perf_counter()

    original_rank = {id(d): i + 1 for i, d in enumerate(candidates)}

    print(f"질문: {query}")
    print(f"벡터 검색({len(candidates)}개): {(t1 - t0) * 1000:.0f}ms / 리랭킹: {(t2 - t1) * 1000:.0f}ms")
    print(f"\n[리랭킹 전] 벡터 검색 상위 {top_n}")
    for i, d in enumerate(candidates[:top_n], 1):
        print(f"  {i}위 (청크 {d.metadata['chunk_id']}) {_preview(d.page_content)}")
    print(f"\n[리랭킹 후] 상위 {top_n}")
    for i, d in enumerate(reranked, 1):
        print(f"  {i}위 <- 원래 {original_rank[id(d)]}위 "
              f"(점수 {d.metadata['rerank_score']:.3f}, 청크 {d.metadata['chunk_id']}) {_preview(d.page_content)}")

In [13]:
test_questions = [
    "AI 에이전트의 주요 속성은 무엇인가요?",
    "NIST는 에이전트형 AI를 어떻게 설명하나요?",
    "에이전트 간 통신을 표준화하기 위해 어떤 프로토콜이 제안되었나요?",
    "AI 에이전트는 주로 어디에 쓰이나요?",
]

for q in test_questions:
    compare(q)
    print("=" * 100)

질문: AI 에이전트의 주요 속성은 무엇인가요?
벡터 검색(20개): 224ms / 리랭킹: 987ms

[리랭킹 전] 벡터 검색 상위 4
  1위 (청크 0) 생성형 인공지능의 맥락에서 AI 에이전트(AI agent, 복합 AI 시스템 또는 에이전트형 AI)는 다양한 정도의 자율성을 
  2위 (청크 19) 예상되는 이점 AI 에이전트는 개인 및 경제적 생산성을 높이고, 더 큰 혁신을 촉진하며, 지루한 작업으로부터 사용자를 해방시키
  3위 (청크 1) AI 에이전트에는 표준화된 정의가 없다. NIST는 에이전트형 AI를 안전한 운영, 상호 운용성 및 외부 시스템과의 신뢰할 수
  4위 (청크 10) 디 인포메이션은 AI 에이전트를 7가지 원형으로 나누었다: 기업용 소프트웨어 내에서 동작하는 비즈니스 작업 에이전트, 고객 지

[리랭킹 후] 상위 4
  1위 <- 원래 1위 (점수 1.000, 청크 0) 생성형 인공지능의 맥락에서 AI 에이전트(AI agent, 복합 AI 시스템 또는 에이전트형 AI)는 다양한 정도의 자율성을 
  2위 <- 원래 5위 (점수 0.878, 청크 6) 4계층: 전개 및 인프라 - AI 에이전트를 실행하기 위한 강력한 기술적 토대를 제공한다. 5계층: 평가 및 관측 가능성 - 
  3위 <- 원래 4위 (점수 0.867, 청크 10) 디 인포메이션은 AI 에이전트를 7가지 원형으로 나누었다: 기업용 소프트웨어 내에서 동작하는 비즈니스 작업 에이전트, 고객 지
  4위 <- 원래 3위 (점수 0.862, 청크 1) AI 에이전트에는 표준화된 정의가 없다. NIST는 에이전트형 AI를 안전한 운영, 상호 운용성 및 외부 시스템과의 신뢰할 수
질문: NIST는 에이전트형 AI를 어떻게 설명하나요?
벡터 검색(20개): 1058ms / 리랭킹: 835ms

[리랭킹 전] 벡터 검색 상위 4
  1위 (청크 1) AI 에이전트에는 표준화된 정의가 없다. NIST는 에이전트형 AI를 안전한 운영, 상호 운용성 및 외부 시스템과의 신뢰할 수

## 6. 최종 답변 비교

In [14]:
for q in test_questions:
    print(f"질문: {q}\n")
    print("[기존]")
    print(baseline_chain.invoke(q))
    print("\n[리랭킹]")
    print(rerank_chain.invoke(q))
    print("=" * 100)

질문: AI 에이전트의 주요 속성은 무엇인가요?

[기존]
AI 에이전트의 주요 속성은 다음과 같습니다: 목표 지향적 행동, 자연어 인터페이스, 외부 도구 사용 능력, 다단계 작업 수행 능력 등이 있습니다. 이들의 제어 흐름은 빈번하게 대형 언어 모델(LLM)에 의해 구동됩니다. (출처: 청크 0)

[리랭킹]
AI 에이전트의 주요 속성은 다음과 같습니다: 목표 지향적 행동, 자연어 인터페이스, 외부 도구 사용 능력, 다단계 작업 수행 능력 등이 있습니다. 이들의 제어 흐름은 빈번하게 대형 언어 모델(LLM)에 의해 구동됩니다. (출처: 청크 0)
질문: NIST는 에이전트형 AI를 어떻게 설명하나요?

[기존]
NIST는 에이전트형 AI를 안전한 운영, 상호 운용성 및 외부 시스템과의 신뢰할 수 있는 상호 작용을 위한 표준이 필요한 신흥 분야로 설명합니다. (청크 1)

[리랭킹]
NIST는 에이전트형 AI를 안전한 운영, 상호 운용성 및 외부 시스템과의 신뢰할 수 있는 상호 작용을 위한 표준이 필요한 신흥 분야로 설명합니다. (청크 1)
질문: 에이전트 간 통신을 표준화하기 위해 어떤 프로토콜이 제안되었나요?

[기존]
에이전트 간 통신을 표준화하기 위해 모델 컨텍스트 프로토콜, Gibberlink 등 여러 프로토콜이 제안되었습니다. (청크 1)

[리랭킹]
에이전트 간 통신을 표준화하기 위해 모델 컨텍스트 프로토콜, Gibberlink 등 여러 프로토콜이 제안되었습니다. (청크 1)
질문: AI 에이전트는 주로 어디에 쓰이나요?

[기존]
AI 에이전트는 주로 작업 자동화에 사용됩니다. 예를 들어, 사용자의 프롬프트 요청에 따라 여행 계획을 예약하는 등의 응용 사례가 있습니다. (청크 1)

[리랭킹]
AI 에이전트는 비디오 게임 개발, 도박(스포츠 베팅 포함), 암호화폐 지갑(암호화폐 거래 및 밈 코인 포함), 소셜 미디어 등에서 사용됩니다. 또한, 소프트웨어 개발, 고객 지원, 비즈니스 작업, 연구, 데이터 분석 등 다양한 분야에서도 활용되고 있습니다

# 웹 chatbot 실행 코드

In [15]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 50.8 MB/s eta 0:00:00


In [19]:
%%writefile app.py
import os
import time
import streamlit as st
import torch
from sentence_transformers import CrossEncoder
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

DOC_PATH = "/content/ai_agent_wiki.txt"
FIRST_STAGE_K = 10   # 1단계 후보 수
FINAL_TOP_N = 4      # LLM에 넘길 최종 청크 수

st.set_page_config(page_title="AI 에이전트 위키 챗봇", page_icon="🤖")


# ---------- 무거운 것들: 한 번만 만들고 재사용 ----------
@st.cache_resource
def load_vectorstore():
    docs = TextLoader(DOC_PATH, encoding="utf-8").load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400, chunk_overlap=80, separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_documents(docs)
    for i, c in enumerate(chunks):
        c.metadata["chunk_id"] = i
    return Chroma.from_documents(
        chunks, OpenAIEmbeddings(model="text-embedding-3-small"), collection_name="ai_agent_app"
    )


@st.cache_resource
def load_reranker():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=device)


@st.cache_resource
def load_chain():
    prompt = ChatPromptTemplate.from_template("""
당신은 제공된 문서를 기반으로 질문에 답변하는 어시스턴트입니다.
아래 제공된 컨텍스트만을 사용하여 질문에 답변하세요.
컨텍스트에 답변할 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.
답변에 사용한 문서의 출처(청크 번호)를 명시하세요.

컨텍스트:
{context}

질문: {question}

답변:
""")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    return prompt | llm | StrOutputParser()


# ---------- 검색 ----------
def retrieve(query, use_rerank):
    vs = load_vectorstore()
    if not use_rerank:
        docs = vs.similarity_search(query, k=FINAL_TOP_N)
        return [(d, None, i + 1) for i, d in enumerate(docs)]

    candidates = vs.similarity_search(query, k=FIRST_STAGE_K)
    scores = load_reranker().predict([(query, d.page_content) for d in candidates])
    ranked = sorted(
        [(d, float(s), i + 1) for i, (d, s) in enumerate(zip(candidates, scores))],
        key=lambda x: x[1], reverse=True,
    )
    return ranked[:FINAL_TOP_N]   # (문서, 리랭킹 점수, 원래 순위)


def show_sources(src):
    with st.expander(f"참고한 청크 · {src['mode']} · 검색 {src['search_ms']:.0f}ms"):
        for s in src["items"]:
            head = f"**청크 {s['chunk_id']}**"
            if s["score"] is not None:
                head += f" · 점수 {s['score']:.3f} · 원래 {s['orig_rank']}위"
            st.markdown(head)
            st.caption(s["preview"])


# ---------- 화면 ----------
st.title("🤖 AI 에이전트 위키 챗봇")

with st.sidebar:
    use_rerank = st.toggle("Reranking 사용", value=True)
    st.caption(f"켜면 후보 {FIRST_STAGE_K}개 → 리랭커 상위 {FINAL_TOP_N}개\n\n끄면 벡터 검색 상위 {FINAL_TOP_N}개")
    if st.button("대화 초기화"):
        st.session_state.messages = []

if "messages" not in st.session_state:
    st.session_state.messages = []

# 지난 대화 다시 그리기 (Streamlit은 입력할 때마다 화면을 처음부터 다시 그림)
for m in st.session_state.messages:
    with st.chat_message(m["role"]):
        st.markdown(m["content"])
        if m.get("sources"):
            show_sources(m["sources"])

if query := st.chat_input("AI 에이전트에 대해 물어보세요"):
    st.session_state.messages.append({"role": "user", "content": query})
    with st.chat_message("user"):
        st.markdown(query)

    with st.chat_message("assistant"):
        with st.spinner("검색하고 답변을 만드는 중..."):
            t0 = time.perf_counter()
            results = retrieve(query, use_rerank)
            search_ms = (time.perf_counter() - t0) * 1000

            context = "\n\n".join(f"[청크 {d.metadata['chunk_id']}]\n{d.page_content}" for d, _, _ in results)
            reply = load_chain().invoke({"context": context, "question": query})

        st.markdown(reply)
        sources = {
            "mode": "Reranking" if use_rerank else "벡터 검색만",
            "search_ms": search_ms,
            "items": [
                {"chunk_id": d.metadata["chunk_id"], "score": s, "orig_rank": r,
                 "preview": d.page_content[:400].replace("\n", " ")}
                for d, s, r in results
            ],
        }
        show_sources(sources)

    st.session_state.messages.append({"role": "assistant", "content": reply, "sources": sources})

Overwriting app.py


In [20]:
!streamlit run app.py --server.port 8501 --server.headless true &> /content/streamlit.log &

In [21]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!nohup ./cloudflared tunnel --url http://localhost:8501 > /content/cloudflared.log 2>&1 &
!sleep 8 && grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /content/cloudflared.log

cloudflared: Text file busy
https://harvest-charity-quickly-bolt.trycloudflare.com
